<div dir="rtl" align="right">
# التخيّلُ الحركيُّ (Motor Imagery)

## نظرةٌ عامّةٌ
يُحلّلُ هذا الدفترُ ظاهرةَ ERD/ERS في بياناتِ التخيلِ الحركيِّ من BNCI2014-001. يَحسبُ طاقةَ نطاقَي mu (8-12 Hz) و beta (13-30 Hz) عندَ أقطابِ C3 و Cz و C4.

## ماذا يفعلُ؟
- يَحمّلُ بياناتِ التخيلِ الحركيِّ (288 محاولة، 22 قناة)
- يَحسبُ طاقةَ النطاقاتِ عندَ C3/Cz/C4
- يُقارنُ طاقةَ اليدِ اليسرى معَ اليدِ اليمنى

## ماذا تَلاحظُ؟
- طاقةُ mu تَنخفضُ (ERD) في القناةِ المُقابلةِ لِليدِ المُتخيّل
- النتائجُ تُظهرُ التباينَ بينَ C3 و C4
</div>

<div dir="rtl" align="right">
## 1. تثبيتُ المكتباتِ
</div>

In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn

<div dir="rtl" align="right">
## 2. تحميلُ البياناتِ وحسابُ الطاقة
</div>

In [ ]:
import numpy as np
from scipy.signal import welch
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery

FS = 250
dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2, fmin=8, fmax=32)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]; labels = labels[mask]

raw = dataset.get_data(subjects=[1])
s1 = raw[1]; sess = list(s1.values())[0]; run = list(sess.values())[0]
ch_names = run.ch_names
c3_idx = ch_names.index('C3'); c4_idx = ch_names.index('C4'); cz_idx = ch_names.index('Cz')

def band_power(epoch, fs, fmin, fmax):
    n_ch = epoch.shape[0]
    powers = np.zeros(n_ch)
    for ch in range(n_ch):
        freqs, psd = welch(epoch[ch, :], fs=fs, nperseg=min(256, epoch.shape[1]))
        mask = (freqs >= fmin) & (freqs <= fmax)
        powers[ch] = np.trapezoid(psd[mask], freqs[mask])
    return powers

mu_left, mu_right, beta_left, beta_right = [], [], [], []
for i, label in enumerate(labels):
    mu = band_power(X[i], FS, 8, 12)
    beta = band_power(X[i], FS, 13, 30)
    vals = [mu[c3_idx], mu[cz_idx], mu[c4_idx]]
    if label == 'left_hand': mu_left.append(vals); beta_left.append([beta[c3_idx],beta[cz_idx],beta[c4_idx]])
    else: mu_right.append(vals); beta_right.append([beta[c3_idx],beta[cz_idx],beta[c4_idx]])

mu_left = np.array(mu_left); mu_right = np.array(mu_right)
beta_left = np.array(beta_left); beta_right = np.array(beta_right)
print(f"Data: {X.shape}, Left: {len(mu_left)}, Right: {len(mu_right)}")

<div dir="rtl" align="right">
## 3. رسمٌ تفاعليٌّ
</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

channels = ['C3', 'Cz', 'C4']
mu_lm = mu_left.mean(axis=0); mu_rm = mu_right.mean(axis=0)
beta_lm = beta_left.mean(axis=0); beta_rm = beta_right.mean(axis=0)

fig = make_subplots(rows=1, cols=2, subplot_titles=('Mu band (8-12 Hz)', 'Beta band (13-30 Hz)'))
x = list(range(3))
fig.add_trace(go.Bar(x=channels, y=mu_lm, name='Left hand', marker_color='steelblue', offsetgroup='1'), row=1, col=1)
fig.add_trace(go.Bar(x=channels, y=mu_rm, name='Right hand', marker_color='coral', offsetgroup='2'), row=1, col=1)
fig.add_trace(go.Bar(x=channels, y=beta_lm, name='Left hand', marker_color='steelblue', offsetgroup='1', showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(x=channels, y=beta_rm, name='Right hand', marker_color='coral', offsetgroup='2', showlegend=False), row=1, col=2)
fig.update_layout(title='ERD/ERS in Motor Imagery', barmode='group', width=1000, height=450)
fig.show()

<div dir="rtl" align="right">
## خلاصةٌ
- ERD يَظهرُ في نطاقَي mu و beta
- التباينُ بينَ C3 و C4 يَعكسُ التخيلَ الحركيَّ لِليدِ المُقابلة
- هذه السماتُ هي أساسُ تصنيفِ التخيلِ الحركيّ
</div>